# Lab 5: Monte Carlo Integration

---

## Overview

This lab explores **Monte Carlo integration**, a powerful numerical method that uses random sampling to approximate definite integrals. This problem is a synthesis of two GPU patterns you've learned:

1. **Embarrassingly parallel**: Each sample point can be evaluated independently
2. **Reduction**: All sample contributions must be summed to produce the final result

Monte Carlo methods are widely used in physics simulations, financial modeling, computer graphics (ray tracing), and machine learning.

## Learning Objectives

By the end of this lab, you will be able to:

1. Understand Monte Carlo integration and its convergence properties ($1/\sqrt{n}$ error)
2. Recognize the "embarrassingly parallel + reduction" pattern
3. Analyze the performance bottleneck of atomic-based reduction
4. Compare atomic reduction vs. shared memory tree reduction
5. Understand numerical precision considerations in parallel accumulation

## 1. Monte Carlo Integration Theory

### The Algorithm

1. Generate $n$ random points $x_i$ uniformly distributed in $[a, b]$
2. Evaluate $f(x_i)$ at each point (given as input $y_i$)
3. Compute the average: $\bar{y} = \frac{1}{n} \sum_{i=1}^{n} y_i$
4. Multiply by interval width: $\text{result} = (b - a) \times \bar{y}$

### Why GPU?

| Property | Benefit for GPU |
|:---------|:----------------|
| Independent samples | No data dependencies between threads |
| Large sample count | More parallelism = better GPU utilization |
| Reduction step | Standard parallel reduction pattern |

### Convergence

The error decreases with more samples:

$$
\text{Error} \propto \frac{1}{\sqrt{n}}
$$

To halve the error, we need 4x more samples.

## 2. Kernel Implementation Analysis

Let us examine the kernel in `main.hip`:

```cpp
__global__ void montecarlo(const double* y_samples, double* result, 
                           double a, double b, int n_samples) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    if (idx >= n_samples) return;

    double tmp = (b - a) * y_samples[idx] / n_samples;

    atomicAdd(result, tmp); 
}
```

### Algorithm Breakdown

| Step | Operation | Parallelism |
|:-----|:----------|:------------|
| 1 | Load $y_i$ | Coalesced memory access |
| 2 | Compute $(b-a) \times y_i / n$ | Fully parallel |
| 3 | Accumulate to result | Atomic (serialized) |

### Exercise 1: Understand the Algorithm

**Question 1**: Why is the division by `n_samples` done in each thread instead of once at the end?

Your answer: ______

**Question 2**: What is the time complexity of the atomic reduction approach?

Your answer: ______

**Question 3**: How could you modify this kernel to use shared memory reduction instead of atomics?

Your answer: ______

## 3. Setup: Generate Test Data

In [1]:
%%bash
# Generate test cases
python3 geninput.py
echo "Test cases generated:"
ls -la testcases/

=== 1.in: 小測資 ===
=== 2.in: N=1 ===
=== 3.in: 數字大1e7～1e9 ===
=== 4.in: 一正一負大測資 ===
=== 5.in: 雙負大測資 ===
=== 6.in: 正常測資 ===
=== 7.in: 超大測資10000000 ===
=== 8.in: 極大測資100000000 ===
  testcases/1.in: 61 bytes
  testcases/2.in: 13 bytes
  testcases/3.in: 12,811 bytes
  testcases/4.in: 878,082 bytes
  testcases/5.in: 877,897 bytes
  testcases/6.in: 877,825 bytes
  testcases/7.in: 83,388,161 bytes
  testcases/8.in: 933,400,064 bytes
Test cases generated:
total 995576
drwxr-xr-x 2 jovyan users      4096 Jun 19 05:06 .
drwxr-xr-x 4 jovyan users      4096 Jun 19 05:06 ..
-rw-r--r-- 1 jovyan users        61 Jun 19 05:06 1.in
-rw-r--r-- 1 jovyan users        13 Jun 19 05:06 2.in
-rw-r--r-- 1 jovyan users     12811 Jun 19 05:06 3.in
-rw-r--r-- 1 jovyan users    878082 Jun 19 05:06 4.in
-rw-r--r-- 1 jovyan users    877897 Jun 19 05:06 5.in
-rw-r--r-- 1 jovyan users    877825 Jun 19 05:06 6.in
-rw-r--r-- 1 jovyan users  83388161 Jun 19 05:06 7.in
-rw-r--r-- 1 jovyan users 933400064 Jun 19 05:07 8.in


## 4. Compile the Program

In [2]:
%%bash
# Compile the Monte Carlo integration program
hipcc -O2 fs_main.hip -o exe_montecarlo
echo "Compilation complete."

fs_main.hip:21:5: warning: ignoring return value of type 'hipError_t' declared with 'nodiscard' attribute [-Wunused-value]
   21 |     hipMalloc(&d_ysamples, n_samples * sizeof(double));
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
fs_main.hip:22:5: warning: ignoring return value of type 'hipError_t' declared with 'nodiscard' attribute [-Wunused-value]
   22 |     hipMalloc(&d_result, sizeof(double));
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
fs_main.hip:24:5: warning: ignoring return value of type 'hipError_t' declared with 'nodiscard' attribute [-Wunused-value]
   24 |     hipMemcpy(d_ysamples, y_samples, n_samples * sizeof(double), hipMemcpyHostToDevice);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
fs_main.hip:25:5: warning: ignoring return value of type 'hipError_t' declared with 'nodiscard' attribute [-Wunused-value]
   25 |     hipMemset(d_result, 0, sizeof(double));
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~

Compilation complete.


## 5. Run Small Test Cases

In [3]:
%%bash
echo "=== Test 1: Small sample (N=8) ==="
cat testcases/1.in
echo "Result:"
./exe_montecarlo testcases/1.in

=== Test 1: Small sample (N=8) ===
0 2 8
4.1805 1.7864 4.7554 3.9983 2.2786 1.1873 4.1222 1.442
Result:
5.937675


In [4]:
%%bash
echo "=== Test 2: Single sample (N=1) ==="
cat testcases/2.in
echo "Result:"
./exe_montecarlo testcases/2.in

=== Test 2: Single sample (N=1) ===
1 1 1
4.7499
Result:
0.000000


### Exercise 2: Manual Calculation

For Test 1, verify the result manually:

Given: $a = 0$, $b = 2$, $n = 8$ samples

Formula: $\text{result} = \frac{(b-a)}{n} \sum_{i=1}^{n} y_i = \frac{2}{8} \times \sum y_i$

Your calculation:
- Sum of y values: ______
- Expected result: ______
- Does it match the program output? ______

## 6. Performance Scaling

In [5]:
%%bash
echo "=== Test 4: N=100,000 ==="
time ./exe_montecarlo testcases/4.in

=== Test 4: N=100,000 ===
751909.082340



real	0m0.072s
user	0m0.037s
sys	0m0.029s


In [6]:
%%bash
echo "=== Test 7: N=10,000,000 ==="
time ./exe_montecarlo testcases/7.in

=== Test 7: N=10,000,000 ===
9996.980601



real	0m0.894s
user	0m0.815s
sys	0m0.066s


In [7]:
%%bash
echo "=== Test 8: N=100,000,000 ==="
time ./exe_montecarlo testcases/8.in

=== Test 8: N=100,000,000 ===
1000009.031397



real	0m8.454s
user	0m8.001s
sys	0m0.344s


### Exercise 3: Scalability Analysis

Record the execution times:

| Test | Sample Count | Execution Time |
|:-----|:-------------|:---------------|
| 4 | 100,000 | ______ |
| 7 | 10,000,000 | ______ |
| 8 | 100,000,000 | ______ |

**Question 1**: When sample count increases 100x (from test 4 to 7), how much does execution time increase?

Your answer: ______

**Question 2**: Is the scaling linear? Why or why not?

Your answer: ______

## 7. Embarrassingly Parallel vs. Reduction

The Monte Carlo kernel has two distinct phases:

### Phase 1: Embarrassingly Parallel

```cpp
double tmp = (b - a) * y_samples[idx] / n_samples;
```

- Each thread computes independently
- No synchronization needed
- Scales perfectly with GPU cores

### Phase 2: Reduction (Bottleneck)

```cpp
atomicAdd(result, tmp);
```

- All threads compete for single location
- Serialized execution
- Does not scale with more threads

### Exercise 4: Bottleneck Analysis

**Question 1**: If the computation takes 1 cycle and atomic takes 100 cycles, what is the theoretical efficiency for N=1M threads?

Hint: Total work = N x (compute + atomic)

Your calculation: ______

**Question 2**: How would shared memory reduction improve this?

Your answer: ______

**Question 3**: With 256 threads per block and N=1M, how many global atomics would shared memory reduction require?

Your calculation: ______

## 8. Numerical Precision

### Floating Point Accumulation

Adding many small numbers to a large sum can cause precision loss:

```
sum = 1000000.0
sum += 0.0001  // May be lost due to precision
```

### Why Tree Reduction Helps

| Method | Accumulation Pattern | Precision |
|:-------|:--------------------|:----------|
| Sequential | Large + small + small + ... | Poor |
| Tree | (a+b) + (c+d) + ... | Better |

### Exercise 5: Precision Considerations

**Question 1**: The kernel uses `double` (64-bit). How many significant digits does double precision provide?

Your answer: ______

**Question 2**: If we switch to `float` (32-bit), what problems might occur with N=100M samples?

Your answer: ______

## 9. Optimized Implementation Design

### Two-Phase Approach

```cpp
__global__ void montecarlo_optimized(const double* y_samples, double* result,
                                      double a, double b, int n_samples) {
    __shared__ double sdata[256];
    
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Phase 1: Each thread accumulates multiple elements
    double local_sum = 0.0;
    for (int i = idx; i < n_samples; i += blockDim.x * gridDim.x) {
        local_sum += y_samples[i];
    }
    sdata[tid] = local_sum;
    __syncthreads();
    
    // Phase 2: Tree reduction in shared memory
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            sdata[tid] += sdata[tid + s];
        }
        __syncthreads();
    }
    
    // Only one atomic per block
    if (tid == 0) {
        atomicAdd(result, (b - a) * sdata[0] / n_samples);
    }
}
```

### Exercise 6: Optimization Analysis

**Question 1**: In the optimized version, how many global atomics occur for N=1M with 256 blocks?

Your answer: ______

**Question 2**: What is the purpose of the grid-stride loop `for (int i = idx; i < n_samples; i += blockDim.x * gridDim.x)`?

Your answer: ______

**Question 3**: Why is `(b - a) / n_samples` multiplication done only by thread 0?

Your answer: ______

## 10. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **Monte Carlo Integration** | Approximate integrals using random sampling |
| **Embarrassingly Parallel** | Independent computations with no data dependencies |
| **Reduction** | Combining many values into one |
| **Atomic Contention** | Performance bottleneck when many threads update same location |

### GPU Suitability

| Aspect | GPU Advantage |
|:-------|:--------------|
| Sample evaluation | Thousands of parallel evaluations |
| Large N required for accuracy | More work = better GPU utilization |
| Memory bandwidth | Coalesced reads of sample array |

### Optimization Takeaways

1. Use shared memory to reduce atomic contention
2. Each thread should process multiple elements
3. Tree reduction improves both performance and precision
4. Number of global atomics = number of blocks (not threads)

## 11. Challenge Exercises

### Challenge 1: Implement Shared Memory Reduction

Modify `main.hip` to use the two-phase approach shown in Section 9. Compare performance.

### Challenge 2: Compute Pi Using Monte Carlo

Classic application: estimate $\pi$ by sampling points in a square and counting how many fall inside a quarter circle.

$$
\pi \approx 4 \times \frac{\text{points inside circle}}{\text{total points}}
$$

### Challenge 3: Error Analysis

For a known integral (e.g., $\int_0^1 x^2 dx = 1/3$):
1. Run with N = 1000, 10000, 100000, 1000000
2. Calculate absolute error for each
3. Verify the $1/\sqrt{n}$ convergence rate

### Challenge 4: Compare float vs double

Modify the kernel to use `float` instead of `double`. Test with large N and compare:
- Accuracy of result
- Execution time

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `main.hip` | Student implementation (stdin input) |
| `fs_main.hip` | Reference implementation (file input) |
| `geninput.py` | Test case generator |
| `Makefile` | Build configuration |